[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/endsem/practice/endsem_practice_2.ipynb)

# Endsem practice paper 2

DA2402 · Data Curation and Visualization · Dr. Arun B Ayyar

Thirteen more problems across the whole course, in the order it was taught, on different data and
mostly different methods from paper 1: reshaping, validation regexes, HTML tables, robots.txt,
cursor pagination, Little's test, time-series gaps, skewed outliers, Isolation Forest and DBSCAN,
Fellegi-Sunter weights, SQL window functions, chained Dash callbacks, and Simpson's paradox.
Nothing here is marked. Each problem names a variable `q1` … `q13`. Put your result in it, run the
cell, then open **Answer** to check yourself. Most answers carry a short written part; try it in a
markdown cell of your own before reading ours.

Every dataset was generated for this paper with its defects planted. The files are the `p2_*` files
in [`endsem/practice/data/`](https://github.com/iitm-da/da2402/tree/master/endsem/practice/data),
written by `make_data2.py`.

**Setup.** Run the next cell first. `fetch(name)` downloads a file once into a local `data/` folder
and returns its path. In Colab, first run `!pip install dash`; the other libraries are already
there.


In [ ]:
import os, re, json, sqlite3, io
import requests
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

URL = "https://raw.githubusercontent.com/iitm-da/da2402/master/endsem/practice/data/"

def fetch(name):
    """Download a data file once into ./data/ and return its local path."""
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        r = requests.get(URL + name)
        r.raise_for_status()
        with open(path, "wb") as fh:
            fh.write(r.content)
    return path

### Q1  pandas · wide to long, and a value in the wrong unit

`p2_store_sales_wide.csv` has monthly sales in lakh for 8 stores, one column per month.

1. List the month columns pandas read as `object`, and say why.
2. Melt to long form (`store`, `month`, `sales_lakh`), convert `sales_lakh` with
   `pd.to_numeric(..., errors="coerce")` and count the missing values.
3. One cell was keyed in rupees. Find it as the value more than 100 times its store's median
   (use `groupby(...).transform("median")`) and convert it to lakh.
4. Compute each store's share of that month's total with `transform("sum")`, and each store's
   best month.

Answer variable `q1`: a dict with `object_columns`, `missing_after_coerce`, `fixed_cell` (store, month), `total_lakh`, `top_share_Dec` ([store, share]) and `best_month` (store → month).

In [ ]:
q1 = ...   # your answer
q1

<details>
<summary><b>Answer</b></summary>

```python
wide = pd.read_csv(fetch("p2_store_sales_wide.csv"))
months = wide.columns[1:]
object_cols = [m for m in months if wide[m].dtype == object]

long = wide.melt(id_vars="store", var_name="month", value_name="sales_lakh")
long["sales_lakh"] = pd.to_numeric(long["sales_lakh"], errors="coerce")     # "-" -> NaN
long["month"] = pd.Categorical(long["month"], categories=list(months), ordered=True)

med = long.groupby("store")["sales_lakh"].transform("median")
suspect = long[long["sales_lakh"] > 100 * med]
long.loc[suspect.index, "sales_lakh"] /= 100000                            # rupees -> lakh

long["share"] = long["sales_lakh"] / long.groupby("month", observed=True)["sales_lakh"].transform("sum")
best = long.loc[long.groupby("store")["sales_lakh"].idxmax(), ["store", "month"]]

q1 = {"object_columns": object_cols,
      "missing_after_coerce": int(long["sales_lakh"].isna().sum()),
      "fixed_cell": (suspect["store"].item(), suspect["month"].item()),
      "total_lakh": round(float(long["sales_lakh"].sum()), 2),
      "top_share_Dec": long[long["month"] == "Dec"].nlargest(1, "share")[["store", "share"]]
                       .round(3).values.tolist()[0],
      "best_month": dict(zip(best["store"], best["month"].astype(str)))}
q1
```

```
{'object_columns': ['Jan', 'Feb', 'Mar'], 'missing_after_coerce': 3, 'fixed_cell': ('OMR', 'Jul'), 'total_lakh': 1387.14, 'top_share_Dec': ['Adyar', 0.154], 'best_month': {'Adyar': 'Jun', 'Anna Nagar': 'Aug', 'Mylapore': 'Jun', 'OMR': 'Jul', 'Porur': 'May', 'T Nagar': 'Jun', 'Tambaram': 'Jun', 'Velachery': 'Aug'}}
```

**Written part.** Porur opened in April and the export wrote `-` for January to March, so those three columns
became text, and `sum()` on them concatenates strings instead of failing. Without the unit fix,
OMR's July would read 1,262,000 lakh and every July share would be OMR's. `transform`
returns one value per row, aligned with the original index, which is what a per-row ratio needs;
`agg` returns one per group.

</details>

### Q2  Regular expressions · validating KYC fields

Each line of `p2_kyc.txt` is `Name | PAN: … | Mobile: … | Email: … | PIN: …`.

1. Pull the four fields out with one pattern using named groups.
2. Validate each field with `re.fullmatch`: PAN is 5 capital letters, 4 digits, 1 capital letter;
   a mobile is 10 digits starting 6-9; a PIN code is 6 digits not starting with 0; an email has a
   local part, `@`, a domain and a top-level domain of at least two letters.
3. Count the invalid values per field and the records valid on all four.

Answer variable `q2`: a dict with `records`, `invalid_per_field` (field → count) and `fully_valid`.

In [ ]:
q2 = ...   # your answer
q2

<details>
<summary><b>Answer</b></summary>

```python
kyc = open(fetch("p2_kyc.txt"), encoding="utf-8").read().splitlines()

FIELD = re.compile(r"PAN: (?P<pan>\S+) \| Mobile: (?P<mobile>\S+) \| "
                   r"Email: (?P<email>.+?) \| PIN: (?P<pin>\S+)$")
RULES = {"pan": r"[A-Z]{5}\d{4}[A-Z]",
         "mobile": r"[6-9]\d{9}",
         "email": r"[\w.+-]+@[\w-]+(\.[\w-]+)*\.[a-z]{2,}",
         "pin": r"[1-9]\d{5}"}

recs = pd.DataFrame([FIELD.search(line).groupdict() for line in kyc])
valid = pd.DataFrame({f: recs[f].map(lambda v, p=p: re.fullmatch(p, v) is not None)
                      for f, p in RULES.items()})

q2 = {"records": len(recs),
      "invalid_per_field": (~valid).sum().to_dict(),
      "fully_valid": int(valid.all(axis=1).sum())}
q2
```

```
{'records': 80, 'invalid_per_field': {'pan': 8, 'mobile': 5, 'email': 8, 'pin': 5}, 'fully_valid': 59}
```

**Written part.** `re.search` with the same patterns would pass a 9-character PAN inside a longer string or an
11-digit mobile, because search only needs the pattern somewhere in the value. `fullmatch` (or
`^…$` with `match`) requires the whole value to fit. The planted errors are a truncated PAN, a PAN
in lower case, a mobile starting with 5, a 9-digit mobile, `at` in place of `@`, a `.c` domain and
a PIN code starting with 0.

</details>

### Q3  Scraping · an HTML results table

`p2_results.html` has one table with a two-row header, constituency names with footnote
markers such as `[a]`, vote counts in Indian digit grouping (`5,01,499`), a dash where a party did
not contest, and a Total row written with Western grouping.

1. Read it with `pd.read_html(io.StringIO(html))` and flatten the header to
   `constituency, A, B, C, winner`.
2. Separate the Total row, strip the footnote markers, and convert the vote columns to numbers.
3. Check that your column sums equal the page's own Total row.

Answer variable `q3`: a dict with `seats`, `footnote_marks_left`, `not_contested` (list), `totals_match` (bool) and `seats_won` (party → seats).

In [ ]:
q3 = ...   # your answer
q3

<details>
<summary><b>Answer</b></summary>

```python
html = open(fetch("p2_results.html"), encoding="utf-8").read()
t = pd.read_html(io.StringIO(html))[0]
t.columns = ["constituency", "A", "B", "C", "winner"]          # flatten the two header rows

page_total = t[t["constituency"] == "Total"]
t = t[t["constituency"] != "Total"].copy()
t["constituency"] = t["constituency"].str.replace(r"\s*\[\w\]$", "", regex=True)
for c in ["A", "B", "C"]:
    t[c] = pd.to_numeric(t[c].astype(str).str.replace(",", ""), errors="coerce")
    page_total[c] = pd.to_numeric(page_total[c].astype(str).str.replace(",", ""))

q3 = {"seats": len(t),
      "footnote_marks_left": int(t["constituency"].str.contains(r"\[").sum()),
      "not_contested": t.loc[t["C"].isna(), "constituency"].tolist(),
      "totals_match": bool((t[["A", "B", "C"]].sum().to_numpy() == page_total[["A", "B", "C"]]
                            .to_numpy()[0]).all()),
      "seats_won": t["winner"].value_counts().to_dict()}
q3
```

```
{'seats': 12, 'footnote_marks_left': 0, 'not_contested': ['Arakkonam'], 'totals_match': True, 'seats_won': {'Party A': 7, 'Party B': 5}}
```

**Written part.** Leaving the Total row in doubles every column sum and adds a 13th "constituency". Removing
commas works for both digit groupings because the grouping is display only. The dash becomes NaN,
not 0: Party C got no votes in Arakkonam because it did not stand there. Checking your sums
against the page's Total is the cheapest test that the parse lost nothing.

</details>

### Q4  Ethics · reading robots.txt with urllib

`p2_robots.txt` is a shop's robots file. Parse it with `urllib.robotparser.RobotFileParser`
(`rp.parse(lines)`) and, as user agent `MyCrawler`, check these paths on `https://shop.example`:
`/products/kettle`, `/cart/checkout`, `/account/orders`, `/account/help/returns`,
`/products?sort=price`, `/accounts`. Also report the crawl delay, and whether `BadBot` may fetch
the home page.

Answer variable `q4`: a dict with `can_fetch` (path → bool), `crawl_delay` and `badbot_home`.

In [ ]:
q4 = ...   # your answer
q4

<details>
<summary><b>Answer</b></summary>

```python
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.parse(open(fetch("p2_robots.txt")).read().splitlines())

paths = ["/products/kettle", "/cart/checkout", "/account/orders", "/account/help/returns",
         "/products?sort=price", "/accounts"]
q4 = {"can_fetch": {p: rp.can_fetch("MyCrawler", "https://shop.example" + p) for p in paths},
      "crawl_delay": rp.crawl_delay("MyCrawler"),
      "badbot_home": rp.can_fetch("BadBot", "https://shop.example/")}
q4
```

```
{'can_fetch': {'/products/kettle': True, '/cart/checkout': False, '/account/orders': False, '/account/help/returns': False, '/products?sort=price': True, '/accounts': False}, 'crawl_delay': 5, 'badbot_home': False}
```

**Written part.** Two of urllib's six answers disagree with the rules Google publishes (RFC 9309).
`/account/help/returns`: urllib applies the first rule that matches, `Disallow: /account`, and says
no; under RFC 9309 the longest match wins, so `Allow: /account/help` permits it. That error is on
the safe side. `/products?sort=price`: urllib does not understand `*` inside a path, so
`Disallow: /*?sort=` matches nothing and it says yes; the site meant no. That error is on the
unsafe side, and a crawler built on urllib would fetch pages the owner excluded. `/accounts` is
blocked by both readings, since `Disallow: /account` is a prefix and `/accounts` starts with it.

</details>

### Q5  APIs · cursor pagination

`p2_listings_1.json` is the first page of an API response. Each page has `results` and a
`next_cursor` naming the next file, `null` on the last page. Each result has a nested `price` and
`seller` and a list of `tags`.

1. Follow `next_cursor` until it is `None`, collecting every result.
2. Flatten with `pd.json_normalize`, count and drop records repeated across pages (by `id`).
3. Report median price per seller, the share of verified sellers, the number of listings with no
   tags, and the most common tag (`explode` the tag lists).

Answer variable `q5`: a dict with `pages`, `records_received`, `duplicates`, `unique`, `columns`, `median_price_by_seller`, `verified_share`, `no_tags`, `top_tag`.

In [ ]:
q5 = ...   # your answer
q5

<details>
<summary><b>Answer</b></summary>

```python
cursor, pages, raw = "p2_listings_1.json", 0, []
while cursor:
    doc = json.load(open(fetch(cursor), encoding="utf-8"))
    raw.extend(doc["results"])
    pages += 1
    cursor = doc["next_cursor"]

lst = pd.json_normalize(raw)                     # price.amount, seller.name, seller.verified ...
dupes = int(lst["id"].duplicated().sum())
lst = lst.drop_duplicates("id")
tags = lst.explode("tags")

q5 = {"pages": pages, "records_received": len(raw), "duplicates": dupes, "unique": len(lst),
      "columns": sorted(lst.columns),
      "median_price_by_seller": lst.groupby("seller.name")["price.amount"].median().to_dict(),
      "verified_share": round(float(lst["seller.verified"].mean()), 3),
      "no_tags": int(tags["tags"].isna().sum()),
      "top_tag": tags["tags"].value_counts().idxmax()}
q5
```

```
{'pages': 3, 'records_received': 58, 'duplicates': 1, 'unique': 57, 'columns': ['id', 'price.amount', 'price.currency', 'seller.name', 'seller.verified', 'tags', 'title'], 'median_price_by_seller': {'AutoHub': 42000.0, 'City Cars': 37000.0, 'Ravi Motors': 58000.0}, 'verified_share': 0.702, 'no_tags': 15, 'top_tag': 'automatic'}
```

**Written part.** Record 5020 arrives on page 1 and again on page 2. Offset-paged APIs do this when a record is
inserted or re-sorted between requests, so deduplicating on the API's id is part of every paging
loop. `explode` turns an empty tag list into one row with NaN, so listings without tags survive
it and can be counted.

</details>

### Q6  Missing values · Little's test, then the logistic

`p2_staff.csv` has `age`, `tenure_yrs` (complete), `income_k` and `spend_k` (incomplete).

1. Implement Little's test as in the course notebook: means and covariance from the available data
   (`df.mean()`, `df.cov()`); for each missingness pattern add n × (pattern mean − overall mean)ᵀ
   Σ⁻¹ (pattern mean − overall mean) on its observed columns; degrees of freedom = total observed
   columns over all patterns − number of columns.
2. Little's test says whether some column breaks MCAR, not which. Fit a logistic regression of
   `income_k`'s missingness on `age` and `tenure_yrs` and report the LR p and the Wald p of each
   predictor.

Answer variable `q6`: a dict with `patterns`, `little_d2`, `df`, `little_p`, `income_LR_p` and `income_wald_p`.

In [ ]:
q6 = ...   # your answer
q6

<details>
<summary><b>Answer</b></summary>

```python
staff = pd.read_csv(fetch("p2_staff.csv"))
cols = ["age", "tenure_yrs", "income_k", "spend_k"]

def littles_mcar_test(df, cols):
    means, cov = df[cols].mean(), df[cols].cov()          # available-data estimates
    d2, observed_total = 0.0, 0
    for pattern, idx in df[cols].isna().groupby(cols).groups.items():
        obs = [c for c, miss in zip(cols, pattern) if not miss]
        diff = df.loc[idx, obs].mean() - means[obs]
        d2 += len(idx) * diff @ np.linalg.inv(cov.loc[obs, obs]) @ diff
        observed_total += len(obs)
    dof = observed_total - len(cols)
    return d2, dof, stats.chi2.sf(d2, dof)

d2, dof, p = littles_mcar_test(staff, cols)

y = staff["income_k"].isna().astype(int)
fit = sm.Logit(y, sm.add_constant(staff[["age", "tenure_yrs"]])).fit(disp=0)

q6 = {"patterns": int(staff[cols].isna().drop_duplicates().shape[0]),
      "little_d2": round(float(d2), 2), "df": dof, "little_p": float(f"{p:.3g}"),
      "income_LR_p": float(f"{fit.llr_pvalue:.3g}"),
      "income_wald_p": {k: float(f"{v:.3g}") for k, v in fit.pvalues.items() if k != "const"}}
q6
```

```
{'patterns': 4, 'little_d2': 28.64, 'df': 8, 'little_p': 0.000367, 'income_LR_p': 4.49e-05, 'income_wald_p': {'age': 0.615, 'tenure_yrs': 0.000782}}
```

**Written part.** Little's test rejects MCAR for the file (d² = 28.64 on 8 df, p = 0.00037). The logistic finds
it in `income_k` (LR p = 4.5e-05), and the Wald tests put it on tenure (p = 0.00078) with age
clearing (p = 0.62), although age and tenure correlate at 0.73. Age's own relation to missingness
goes through tenure, and the Wald test asks what each predictor adds given the other. MAR on tenure
is what was planted; the data could not have ruled out MNAR on income.

</details>

### Q7  Imputation · gaps in a time series

`p2_sensor_hourly.csv` is a week of hourly temperatures with single dropped readings and one
12-hour outage. `p2_sensor_hourly_truth.csv` has the full series.

1. Label each run of consecutive missing values and its length (`miss.ne(miss.shift()).cumsum()`
   gives a run id).
2. Fill four ways: series mean, forward fill, linear interpolation, and the same hour the day
   before (`shift(24)`).
3. Report RMSE against the truth separately on single gaps and on the outage.

Answer variable `q7`: a DataFrame with one row per method and columns `rmse_single_gaps`, `rmse_12h_outage`.

In [ ]:
q7 = ...   # your answer
q7

<details>
<summary><b>Answer</b></summary>

```python
s = pd.read_csv(fetch("p2_sensor_hourly.csv"), parse_dates=["time"])
truth = pd.read_csv(fetch("p2_sensor_hourly_truth.csv"))["temp_c"]
miss = s["temp_c"].isna()

run = miss.ne(miss.shift()).cumsum()                     # label each run of missing values
gap_len = miss.groupby(run).transform("sum").where(miss, 0)
short, long_ = miss & (gap_len == 1), miss & (gap_len > 1)

fills = {"mean": s["temp_c"].fillna(s["temp_c"].mean()),
         "ffill": s["temp_c"].ffill(),
         "linear": s["temp_c"].interpolate(method="linear"),
         "same hour yesterday": s["temp_c"].fillna(s["temp_c"].shift(24))}

rmse = lambda f, m: round(float(np.sqrt(((f[m] - truth[m]) ** 2).mean())), 3)
q7 = pd.DataFrame({name: {"rmse_single_gaps": rmse(f, short), "rmse_12h_outage": rmse(f, long_)}
                   for name, f in fills.items()}).T
q7
```

```
                     rmse_single_gaps  rmse_12h_outage
mean                            2.664            2.617
ffill                           0.780            4.428
linear                          0.153            0.680
same hour yesterday             0.719            0.513
```

**Written part.** Linear interpolation wins on single gaps (0.15), because the readings either side of one hour
pin it down. Across the 12-hour outage it draws a straight line through a daily cycle, and the
same hour yesterday does better (0.51 against 0.68) because it carries the cycle. Forward fill is
worst on the outage (4.43): it holds the last reading flat for half a day. The mean ignores time
altogether. The choice depends on gap length, so measure gap lengths before choosing a method.

</details>

### Q8  Outliers · rules on a skewed column

`p2_incomes.csv` has 500 household incomes in lakh per year: right-skewed, with five planted
errors: two values 1,000 times too small, two 100 times too large, and one junk value.
`p2_incomes_truth.csv` marks them.

Apply the IQR rule (1.5) and the modified z-score (3.5) to the raw column and to `log10` of it.
For each of the four, count flags, planted errors caught and clean values flagged.

Answer variable `q8`: a DataFrame with one row per rule and columns `flagged`, `planted_caught`, `clean_flagged`.

In [ ]:
q8 = ...   # your answer
q8

<details>
<summary><b>Answer</b></summary>

```python
inc = pd.read_csv(fetch("p2_incomes.csv"))
planted = pd.read_csv(fetch("p2_incomes_truth.csv"))["planted"]
x = inc["income_lakh"]

def iqr_flags(v):
    q1, q3 = v.quantile([.25, .75])
    return (v < q1 - 1.5 * (q3 - q1)) | (v > q3 + 1.5 * (q3 - q1))

def modz_flags(v):
    med = v.median()
    return (0.6745 * (v - med) / (v - med).abs().median()).abs() > 3.5

rules = {"IQR raw": iqr_flags(x), "IQR log10": iqr_flags(np.log10(x)),
         "modified z raw": modz_flags(x), "modified z log10": modz_flags(np.log10(x))}
q8 = pd.DataFrame({k: {"flagged": int(f.sum()), "planted_caught": int((f & planted).sum()),
                       "clean_flagged": int((f & ~planted).sum())} for k, f in rules.items()}).T
q8
```

```
                  flagged  planted_caught  clean_flagged
IQR raw                20               3             17
IQR log10               6               5              1
modified z raw         12               3              9
modified z log10        5               5              0
```

**Written part.** On the raw scale both rules assume a roughly symmetric middle and flag the long right tail of
honest high earners (17 and 9 clean flags) while missing the two tiny values, which sit inside the
lower fence. On the log scale incomes are close to symmetric, the unit slips sit orders of magnitude
away on both sides, and the modified z catches all five with no clean flag. Transforming first is an
assumption about the shape of the clean data, so check that shape with a histogram before
trusting it.

</details>

### Q9  Outliers · Isolation Forest and DBSCAN

`p2_transactions.csv` has card transactions (`hour`, `amount`); `p2_transactions_truth.csv`
marks 12 planted frauds: 9 large night-time payments and 3 night-time payments of ordinary size.

1. Build features `hour` and `log10(amount)`, standardised.
2. Fit `IsolationForest(n_estimators=200, contamination=0.02, random_state=42)`; flagged = predict
   returns −1.
3. Fit `DBSCAN(eps=0.5, min_samples=8)`; flagged = label −1 (noise).
4. Score precision and recall for each.

Answer variable `q9`: a DataFrame with rows `isolation forest`, `DBSCAN noise` and columns `flagged`, `true_pos`, `precision`, `recall`.

In [ ]:
q9 = ...   # your answer
q9

<details>
<summary><b>Answer</b></summary>

```python
tx = pd.read_csv(fetch("p2_transactions.csv"))
fraud = pd.read_csv(fetch("p2_transactions_truth.csv"))["fraud"]
Z = StandardScaler().fit_transform(np.c_[tx["hour"], np.log10(tx["amount"])])

iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42).fit(Z)
db = DBSCAN(eps=0.5, min_samples=8).fit(Z)

def score(flag):
    tp = int((flag & fraud).sum())
    return {"flagged": int(flag.sum()), "true_pos": tp,
            "precision": round(tp / flag.sum(), 3), "recall": round(tp / fraud.sum(), 3)}

q9 = pd.DataFrame({"isolation forest": score(pd.Series(iso.predict(Z) == -1)),
                   "DBSCAN noise": score(pd.Series(db.labels_ == -1))}).T
q9
```

```
                  flagged  true_pos  precision  recall
isolation forest     13.0      11.0      0.846   0.917
DBSCAN noise         20.0      12.0      0.600   1.000
```

**Written part.** Isolation Forest misses one of the ordinary-sized night payments (3:26 am, ₹505), which is
isolated only in `hour`, and `contamination=0.02` fixes the flag count at 2% of rows whatever the
data holds. DBSCAN catches all 12 but calls 8 honest transactions noise, early-morning and late-night
payments and the smallest and largest daytime amounts, where the density thins. Both depend on a setting the data
cannot choose: the contamination rate, or eps and min_samples.

</details>

### Q10  Duplicates · Fellegi-Sunter weights

`p2_pairs.csv` has 400 candidate record pairs, each with 0/1 agreement on `surname`, `dob`,
`pincode` and `phone`. Use these m and u probabilities:

| field | m | u |
|---|---|---|
| surname | 0.95 | 0.05 |
| dob | 0.90 | 0.003 |
| pincode | 0.85 | 0.10 |
| phone | 0.80 | 0.001 |

1. Agreement weight log₂(m/u), disagreement weight log₂((1−m)/(1−u)), per field.
2. Score each pair as the sum of its four weights.
3. With `p2_pairs_truth.csv`, report flagged pairs, precision and recall at thresholds of 0, 5,
   10 and 14 bits.

Answer variable `q10`: a tuple `(weights, table)`: weights per field (agree, disagree) and the threshold table.

In [ ]:
q10 = ...   # your answer
q10

<details>
<summary><b>Answer</b></summary>

```python
pairs = pd.read_csv(fetch("p2_pairs.csv"))
match = pd.read_csv(fetch("p2_pairs_truth.csv"))["match"]
m = {"surname": .95, "dob": .9, "pincode": .85, "phone": .8}
u = {"surname": .05, "dob": .003, "pincode": .1, "phone": .001}

weights = pd.DataFrame({f: {"agree": np.log2(m[f] / u[f]),
                            "disagree": np.log2((1 - m[f]) / (1 - u[f]))} for f in m}).T.round(3)
pairs["score"] = sum(np.where(pairs[f] == 1, weights.loc[f, "agree"], weights.loc[f, "disagree"])
                     for f in m)

res = {}
for thr in [0, 5, 10, 14]:
    flag = pairs["score"] > thr
    tp = int((flag & match).sum())
    res[f"> {thr} bits"] = {"flagged": int(flag.sum()), "precision": round(tp / flag.sum(), 3),
                            "recall": round(tp / match.sum(), 3)}
q10 = (weights, pd.DataFrame(res).T)
q10
```

```
(         agree  disagree
surname  4.248    -4.248
dob      8.229    -3.318
pincode  3.087    -2.585
phone    9.644    -2.320,            flagged  precision  recall
> 0 bits     133.0      0.887   0.983
> 5 bits     129.0      0.907   0.975
> 10 bits    115.0      0.896   0.858
> 14 bits     74.0      1.000   0.617)
```

**Written part.** Agreeing on phone is worth 9.6 bits and on surname 4.2, because a shared surname is far more
common by chance (u = 0.05). Two groups decide the thresholds. Eight true matches moved and changed
phone; they agree on surname and dob only and score 7.57, so a threshold of 10 loses them. Twelve
family pairs share surname, pincode and phone and score 13.66, above 10. The model treats fields as
independent given match status; in a family they agree together, so the score overstates the
evidence. Only the 14-bit threshold removes them, at a recall of 0.617. A first-name field would
separate them.

</details>

### Q11  Storage · normalising into SQLite, then a window function

`p2_order_lines.csv` repeats each product's name, category and price on every line.

1. Split it into `products` (one row per product) and `order_lines` (line_id, order_date,
   product_id, qty). Create both tables in an in-memory SQLite database with primary keys and a
   foreign key, and load them.
2. How many cells does the split save?
3. SQL: months (`strftime('%Y-%m', order_date)`) with revenue over 1.5 lakh, using `HAVING`.
4. SQL: the top product by revenue in each category with `ROW_NUMBER() OVER (PARTITION BY …)`.

Answer variable `q11`: a dict with `cells_saved`, `months_over_1.5_lakh` (list) and `top_per_category` (category → product).

In [ ]:
q11 = ...   # your answer
q11

<details>
<summary><b>Answer</b></summary>

```python
lines = pd.read_csv(fetch("p2_order_lines.csv"))
products = lines[["product_id", "product", "category", "unit_price"]].drop_duplicates()
order_lines = lines[["line_id", "order_date", "product_id", "qty"]]

con = sqlite3.connect(":memory:")
con.execute("""CREATE TABLE products (product_id TEXT PRIMARY KEY, product TEXT,
               category TEXT, unit_price INTEGER)""")
con.execute("""CREATE TABLE order_lines (line_id INTEGER PRIMARY KEY, order_date TEXT,
               product_id TEXT REFERENCES products(product_id), qty INTEGER)""")
products.to_sql("products", con, if_exists="append", index=False)
order_lines.to_sql("order_lines", con, if_exists="append", index=False)

big_months = pd.read_sql("""
    SELECT strftime('%Y-%m', order_date) AS month, SUM(qty * unit_price) AS revenue
    FROM order_lines JOIN products USING (product_id)
    GROUP BY month HAVING revenue > 150000 ORDER BY month""", con)

top_per_cat = pd.read_sql("""
    SELECT category, product, revenue FROM (
        SELECT category, product, SUM(qty * unit_price) AS revenue,
               ROW_NUMBER() OVER (PARTITION BY category ORDER BY SUM(qty * unit_price) DESC) AS rk
        FROM order_lines JOIN products USING (product_id)
        GROUP BY category, product)
    WHERE rk = 1 ORDER BY category""", con)

q11 = {"cells_saved": lines.size - products.size - order_lines.size,
       "months_over_1.5_lakh": big_months["month"].tolist(),
       "top_per_category": dict(zip(top_per_cat["category"], top_per_cat["product"]))}
q11
```

```
{'cells_saved': 1464, 'months_over_1.5_lakh': ['2025-07', '2025-08', '2025-10'], 'top_per_category': {'Electronics': 'Earphones', 'Kitchen': 'Tawa', 'Stationery': 'Planner'}}
```

**Written part.** `WHERE` filters rows before grouping and cannot see `SUM(...)`; `HAVING` filters the groups.
The window function ranks within each category without collapsing rows, which a `GROUP BY
category` with `MAX` cannot do, since `MAX` returns the revenue and not the product that earned it.
Normalising also means a price change is one update in `products` instead of hundreds of lines
that can disagree.

</details>

### Q12  Dash · chained callbacks, State and PreventUpdate

With Gapminder (`px.data.gapminder()`):

1. A continent dropdown (`cont`) fills the options and first value of a country dropdown
   (`country`).
2. A button (`go`) shows the life expectancy for the chosen country and a year typed in an input
   (`year`). The country and year are `State`, so typing does not fire the callback; only the
   click does.
3. Raise `PreventUpdate` before the first click. If the year has no row, say so instead of
   failing.

Test both callbacks as plain functions: `countries("Oceania")`, then `show(None, "India", 2007)`
(should raise), `show(1, "India", 2007)` and `show(1, "India", 2005)`.

Answer variable `q12`: a tuple `(options, first_value, prevent_raised, text_2007, text_2005)`.

In [ ]:
q12 = ...   # your answer
q12

<details>
<summary><b>Answer</b></summary>

```python
from dash import Dash, dcc, html, Input, Output, State
from dash.exceptions import PreventUpdate
import plotly.express as px

gm = px.data.gapminder()
app = Dash(__name__)
app.layout = html.Div([
    dcc.Dropdown(id="cont", options=sorted(gm["continent"].unique()), value="Asia"),
    dcc.Dropdown(id="country"),
    dcc.Input(id="year", type="number", value=2007),
    html.Button("Show", id="go"),
    html.Div(id="out")])

@app.callback(Output("country", "options"), Output("country", "value"), Input("cont", "value"))
def countries(cont):
    opts = sorted(gm.loc[gm["continent"] == cont, "country"].unique())
    return opts, opts[0]

@app.callback(Output("out", "children"), Input("go", "n_clicks"),
              State("country", "value"), State("year", "value"))
def show(n_clicks, country, year):
    if not n_clicks:
        raise PreventUpdate
    row = gm[(gm["country"] == country) & (gm["year"] == year)]
    if row.empty:
        return f"No Gapminder row for {country} in {year} (data runs 1952-2007 in steps of 5)."
    return f"{country}, {year}: life expectancy {row['lifeExp'].item():.1f} years"

# app.run(jupyter_mode="inline")      # uncomment to see the app
opts, first = countries("Oceania")
try:
    show(None, "India", 2007)
    blocked = False
except PreventUpdate:
    blocked = True
q12 = (opts, first, blocked, show(1, "India", 2007), show(1, "India", 2005))
q12
```

```
(['Australia', 'New Zealand'], 'Australia', True, 'India, 2007: life expectancy 64.7 years', 'No Gapminder row for India in 2005 (data runs 1952-2007 in steps of 5).')
```

**Written part.** With `Input` on the year, every keystroke would fire the callback and look up years like 2, 20
and 200. `State` passes the current value only when the button fires. Gapminder has rows every five
years, so 2005 does not exist; a callback that indexes `.item()` on an empty selection raises an
error inside the app, which is why the empty case is handled first.

</details>

### Q13  Honest charts · Simpson's paradox

`p2_recovery.csv` has made-up patients and recoveries by hospital and severity.

1. Recovery rate per hospital overall.
2. Recovery rate per hospital within each severity.
3. Standardise: weight each hospital's severity-specific rates by the pooled severity mix of all
   2,000 patients.
4. A bubble chart scaled the bubble **radius** by value. How many times larger does a bubble for
   twice the value look, in area?

Answer variable `q13`: a dict with `overall`, `by_severity`, `standardised` and `area_ratio_if_radius_scaled_for_2x`.

In [ ]:
q13 = ...   # your answer
q13

<details>
<summary><b>Answer</b></summary>

```python
rec = pd.read_csv(fetch("p2_recovery.csv"))
rec["rate"] = rec["recovered"] / rec["patients"]

overall = rec.groupby("hospital")[["recovered", "patients"]].sum()
overall = (overall["recovered"] / overall["patients"]).round(3).to_dict()
by_sev = rec.pivot(index="severity", columns="hospital", values="rate").round(3)

mix = rec.groupby("severity")["patients"].sum() / rec["patients"].sum()     # pooled case mix
standardised = (by_sev.mul(mix, axis=0).sum()).round(3).to_dict()

# a bubble chart that scales the radius by value: 2x the value draws 4x the area
q13 = {"overall": overall, "by_severity": by_sev.to_dict(), "standardised": standardised,
       "area_ratio_if_radius_scaled_for_2x": 2 ** 2}
q13
```

```
{'overall': {'City Hospital': 0.85, 'Town Clinic': 0.888}, 'by_severity': {'City Hospital': {'mild': 0.95, 'severe': 0.7}, 'Town Clinic': {'mild': 0.92, 'severe': 0.6}}, 'standardised': {'City Hospital': 0.888, 'Town Clinic': 0.84}, 'area_ratio_if_radius_scaled_for_2x': 4}
```

**Written part.** Town Clinic looks better overall (88.8% against 85.0%) and City Hospital is better at both
severities (95% vs 92% mild, 70% vs 60% severe). City takes 400 severe cases out of 1,000; Town
takes 100. Standardised to one case mix, City leads 88.8% to 84.0%. A chart of overall rates
answers "where did more patients recover"; small multiples or grouped bars by severity answer
"which hospital treats better". Scaling a bubble's radius draws a 2× value at 4× the area, and
readers judge area, so area should be proportional to the value. `px.scatter(size=...)` does
this (it sets `sizemode="area"`); in `graph_objects` the default sizemode is `"diameter"`.

</details>

That is paper 2. Errors you spot in it: tell Dr. Ayyar in class or on Moodle.